Import additional libraries needed

In [32]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, RobustScaler

from imblearn.over_sampling import SMOTE

In [33]:
df = pd.read_csv('./dataset/creditcard.csv')

In [34]:
df.head(5)

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,V11,V12,V13,V14,V15,V16,V17,V18,V19,V20,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,0.090794,-0.551600,-0.617801,-0.991390,-0.311169,1.468177,-0.470401,0.207971,0.025791,0.403993,0.251412,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,-0.166974,1.612727,1.065235,0.489095,-0.143772,0.635558,0.463917,-0.114805,-0.183361,-0.145783,-0.069083,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,0.207643,0.624501,0.066084,0.717293,-0.165946,2.345865,-2.890083,1.109969,-0.121359,-2.261857,0.524980,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,-0.054952,-0.226487,0.178228,0.507757,-0.287924,-0.631418,-1.059647,-0.684093,1.965775,-1.232622,-0.208038,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,0.753074,-0.822843,0.538196,1.345852,-1.119670,0.175121,-0.451449,-0.237033,-0.038195,0.803487,0.408542,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


Split the output and input feature

In [35]:
X = df.drop(columns=['Class'])
y = df['Class']

In [36]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [37]:
print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("Train fraud ratio:", y_train.mean())
print("Test fraud ratio:", y_test.mean())

Train shape: (227845, 30)
Test shape: (56962, 30)
Train fraud ratio: 0.001729245759178389
Test fraud ratio: 0.0017204452090867595


In [38]:
scaler = RobustScaler()

X_train['Amount_scaled'] = scaler.fit_transform(X_train[['Amount']])
X_test['Amount_scaled'] = scaler.fit_transform(X_test[['Amount']])

X_train['Time_scaled'] = scaler.fit_transform(X_train[['Time']])
X_test['Time_scaled'] = scaler.fit_transform(X_test[['Time']])

X_train = X_train.drop(columns=['Amount', 'Time'])
X_test = X_test.drop(columns=['Amount', 'Time'])

In [39]:
X_train['Hour'] = (df.loc[X_train.index, 'Time'] // 3600) % 24
X_test['Hour'] = (df.loc[X_test.index, 'Time'] // 3600) % 24

In [40]:
train_corr = pd.concat([X_train, y_train], axis=1).corr()['Class'].abs().sort_values(ascending=False)
print(train_corr)

weak_features = train_corr[train_corr < 0.02].index.tolist()
weak_features = [f for f in weak_features if f != 'Class']
print("Weak features to consider dropping:", weak_features)

Class            1.000000
V17              0.321937
V14              0.301054
V12              0.259989
V10              0.217894
V3               0.194135
V16              0.193826
V7               0.186184
V11              0.153709
V4               0.135014
V18              0.108732
V1               0.100041
V9               0.098247
V5               0.093578
V2               0.090586
V6               0.043334
V21              0.035588
V19              0.032380
V8               0.020552
V20              0.019385
Hour             0.016952
V27              0.016034
Time_scaled      0.010564
V28              0.009810
V24              0.007483
Amount_scaled    0.006211
V13              0.005861
V15              0.005705
V23              0.004169
V26              0.004119
V22              0.002926
V25              0.001618
Name: Class, dtype: float64
Weak features to consider dropping: ['V20', 'Hour', 'V27', 'Time_scaled', 'V28', 'V24', 'Amount_scaled', 'V13', 'V15', 'V23', 'V26', 'V22', 

In [41]:
X_train.to_csv('X_train.csv', index=False)
y_train.to_csv('y_train.csv', index=False)
X_test.to_csv('X_test.csv', index=False)
y_test.to_csv('y_test.csv', index=False)

print("Processed datasets saved")

Processed datasets saved


### Feature Engineering Summary

- Split data into train/test **before** scaling or resampling to avoid data leakage.
- Scaled `Amount` and `Time` using `RobustScaler` due to their skewed distributions and outliers.
- Engineered an `Hour` feature from `Time` to capture time-of-day patterns.
- Evaluated feature correlations with `Class` on training data only; optionally dropped weakly correlated features.
- Applied SMOTE to the training set only, to address the severe class imbalance identified during EDA.
- Saved processed train/test sets for the modeling stage.

**Next step:** Model training and evaluation (e.g., Logistic Regression, Random Forest, XGBoost), using appropriate metrics for imbalanced classification such as Precision, Recall, F1, and AUC-PR rather than accuracy.